# Example Usage of Metics in TopicGPT: 20 Newsgroups Dataset

In this notebook, we will use the 20 Newsgroups dataset to demonstrate the use of the topicgpt package

In [ ]:
import sys
import os

# Add the 'src' directory to the Python path
sys.path.append(os.path.abspath("../src"))

### Configurations

TRAIN = True
PROVIDER = "anthropic"  # "openai" or "anthropic" or "gemini"
EMBEDDING_PATH = "./SavedEmbeddings/{PROVIDER}_news.pkl"


In [ ]:
# select your own API key here. (Note: This specific code will not work for you unless you specified an environment variable for OPENAI_API_KEY)
import os
from dotenv import load_dotenv

load_dotenv()

if PROVIDER == "openai":
    api_key = os.environ.get('OPENAI_API_KEY')
    prompting_model = "gpt-3.5-turbo"
    embedding_model = "text-embedding-3-small"
elif PROVIDER == "gemini":
    api_key = os.environ.get('GEMINI_API_KEY')
    prompting_model = "gemini-2.0-flash-lite"
    embedding_model = "gemini-embedding-001"
elif PROVIDER == "anthropic":
    api_key = os.environ.get('ANTHROPIC_API_KEY')
    prompting_model = "claude-sonnet-4-20250514"
    embedding_model = "all-MiniLM-L6-v2"

### Load Data

In [ ]:
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes')) #download the 20 Newsgroups dataset
corpus = data['data']
corpus = [doc for doc in corpus if doc != ""]

## Initialize and fit the model 

In [ ]:
from topicgpt.TopicGPT import TopicGPT
if TRAIN:
    tm = TopicGPT(
        prompting_model=prompting_model,
        api_key=api_key,
        n_topics=20,  # select 20 topics since the true number of topics is 20
        embedding_model=embedding_model, 
        use_saved_embeddings=False,  # set to False to train the model from scratch
    )
    tm.fit(corpus)  # train the model on the corpus
    tm.save_embeddings(EMBEDDING_PATH) #save the embeddings for future use
else:
    tm = TopicGPT(
        api_key=api_key,
        n_topics=20,  # select 20 topics since the true number of topics is 20
        path_saved_embeddings=EMBEDDING_PATH,
        use_saved_embeddings=True,  # set to True to use saved embeddings
    )

In [ ]:
tm

## Get an overview over the identified topics

In [ ]:
### Some information about the trained model
print(f'Number of documents: {len(corpus)}')
print(f'document embedding shape: {tm.document_embeddings.shape}')
print(f'number of vocab: {len(list(tm.vocab_embeddings.keys()))}')
print(f'vocab embedding shape: {list(tm.vocab_embeddings.values())[0].shape}') #shape of a single vocab embedding

In [ ]:
# We need the list of topics to compute the metrics which is done extract_topics()
tm.extract_topics(corpus)

In [ ]:
tm.score()